# SMILES 分子结构与原子编号查看器

在下方输入 SMILES，即可查看分子的二维结构与 **从 0 开始** 的原子编号。填写 `cores.xlsx` 的 `sub_idx` 时可使用这些编号。

需要的环境：`rdkit`、`ipywidgets` 和 Jupyter Notebook/Lab。

In [1]:
from IPython.display import Image, display
from ipywidgets import Text, interact
from rdkit import Chem
from rdkit.Chem import rdMolDraw2D


ImportError: cannot import name 'rdMolDraw2D' from 'rdkit.Chem' (D:\anaconda3\envs\env\Lib\site-packages\rdkit\Chem\__init__.py)

In [ ]:
def show_molecule(smiles: str, width: int = 600, height: int = 400) -> None:
    """绘制 SMILES，并标出 RDKit 的 0 起始原子编号。"""
    smiles = smiles.strip()
    if not smiles:
        print('请输入 SMILES。')
        return

    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        print('无法解析该 SMILES，请检查括号、环编号与原子符号。')
        return

    drawer = rdMolDraw2D.MolDraw2DCairo(width, height)
    options = drawer.drawOptions()
    options.addAtomIndices = True
    options.padding = 0.08
    rdMolDraw2D.PrepareAndDrawMolecule(drawer, molecule)
    drawer.FinishDrawing()
    display(Image(data=drawer.GetDrawingText()))

    print('原子编号（从 0 开始）：')
    for atom in molecule.GetAtoms():
        print(f'{atom.GetIdx():>3}: {atom.GetSymbol()}')


## 单次查看

修改下一单元格中的 `smiles` 后运行即可。

In [ ]:
smiles = 'c1ccccc1'  # 示例：苯
show_molecule(smiles)

## 交互式输入

运行下一单元格后，直接在输入框中粘贴或键入 SMILES；按 Enter 更新图形。

In [ ]:
smiles_input = Text(
    value='CC(=O)O',
    placeholder='例如：c1ccccc1',
    description='SMILES:',
    layout={'width': '600px'},
)
interact(show_molecule, smiles=smiles_input);

# 芳香环可取代位点（含对称位点）

下列函数仅识别可用于芳香环取代的位点：**芳香碳原子且相邻原子数为 2**。这会排除已经连接取代基的芳香碳，以及稠环连接处的芳香碳。

对称性由 RDKit 的 canonical rank 判定。输出中 `(1,5)` 代表一组对称位点；单独的数字代表不与其他候选位点对称的位点。

In [ ]:
def find_substitution_sites(molecule):
    """返回芳香环上可取代的碳原子索引（从 0 开始）。"""
    return sorted(
        atom.GetIdx()
        for atom in molecule.GetAtoms()
        if atom.GetAtomicNum() == 6
        and atom.GetIsAromatic()
        and atom.GetDegree() == 2
    )


def group_symmetric_sites(molecule, site_indices=None):
    """按分子对称性对候选位点分组，返回由索引元组组成的列表。"""
    if site_indices is None:
        site_indices = find_substitution_sites(molecule)

    ranks = Chem.CanonicalRankAtoms(molecule, breakTies=False)
    groups_by_rank = {}
    for atom_index in site_indices:
        groups_by_rank.setdefault(ranks[atom_index], []).append(atom_index)

    return sorted(
        (tuple(indices) for indices in groups_by_rank.values()),
        key=lambda group: group[0],
    )


def format_site_groups(site_groups):
    """将位点分组格式化为适合填写 sub_idx 的文本。"""
    return ', '.join(
        f"({', '.join(map(str, group))})" if len(group) > 1 else str(group[0])
        for group in site_groups
    )


def find_substitutable_aromatic_sites(smiles):
    """解析 SMILES，返回（候选位点列表、对称分组、sub_idx 格式文本）。"""
    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        raise ValueError('无法解析该 SMILES。')

    sites = find_substitution_sites(molecule)
    groups = group_symmetric_sites(molecule, sites)
    return sites, groups, format_site_groups(groups)


In [ ]:
def show_substitution_sites(smiles: str, width: int = 600, height: int = 400) -> None:
    """显示可取代位点、原子编号及对称位点分组。"""
    smiles = smiles.strip()
    if not smiles:
        print('请输入 SMILES。')
        return

    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        print('无法解析该 SMILES，请检查括号、环编号与原子符号。')
        return

    sites = find_substitution_sites(molecule)
    groups = group_symmetric_sites(molecule, sites)

    drawer = rdMolDraw2D.MolDraw2DCairo(width, height)
    options = drawer.drawOptions()
    options.addAtomIndices = True
    options.padding = 0.08
    rdMolDraw2D.PrepareAndDrawMolecule(
        drawer, molecule, highlightAtoms=sites, highlightAtomColors={idx: (0.55, 0.85, 0.55) for idx in sites}
    )
    drawer.FinishDrawing()
    display(Image(data=drawer.GetDrawingText()))

    print(f'可取代位点：{sites}')
    print(f'对称位点分组：{format_site_groups(groups) or "无"}')
    print('可直接填写到 sub_idx：', format_site_groups(groups))


运行下一单元格并输入 SMILES。图中的绿色原子为识别出的可取代位点。

In [ ]:
substitution_smiles_input = Text(
    value='c1ccccc1',
    placeholder='例如：c1ccccc1',
    description='SMILES:',
    layout={'width': '600px'},
)
interact(show_substitution_sites, smiles=substitution_smiles_input);